
# **Step 1:** The Ingestion Block (Connecting & Extracting)

This block sets up a secure connection socket to your locally running MetaTrader 5 terminal, requests a clean slice of historical time-series data, and converts the raw binary structure into a localized Pandas DataFrame.
Create a new cell and write the following code:

In [2]:
import MetaTrader5 as mt5 
import pandas as pd 
import numpy as np

def connect_and_fetch_forex_data(symbol:str, timeframe,num_bars:int) -> pd.DataFrame:
    if not mt5. initialize():
        raise RuntimeError(f"Could not Connect to MT5. \n Error: {mt5.last_error()}")

    print("Initialized Connection to MT5 Terminal.....")

    mt5.symbol_select(symbol,True)

    raw_rates = mt5.copy_rates_from_pos(symbol,timeframe,0,num_bars)

    mt5.shutdown()
    print("MT5 Terminal Closed Successfully!")

    df = pd.DataFrame(raw_rates)

    df['time'] = pd.to_datetime(df['time'],unit='s')
    df.set_index('time',inplace=True)

    essential_columns = ['open', 'high', 'low', 'close', 'tick_volume']
    return df[essential_columns]

raw_eurusd_df =  connect_and_fetch_forex_data("EURUSD",mt5.TIMEFRAME_H1,5000)
print(f"Matrix Shape Extracted: {raw_eurusd_df.shape}")

Initialized Connection to MT5 Terminal.....
MT5 Terminal Closed Successfully!
Matrix Shape Extracted: (5000, 5)


# **Step 2 :** Feature Enginering, here I am gonna take the raw rates and then create the Engineered stationary features.

In [3]:
def engineer_stationary_features(raw_df: pd.DataFrame) -> pd.DataFrame: 
    matrix = raw_df.copy()

    matrix['log_return'] = np.log(matrix['close']/matrix['close'].shift(1))

    matrix['high_low_span'] = matrix['high'] - matrix['low']
    matrix['volatility_atr'] = matrix['high_low_span'].rolling(window=14).mean()

    matrix['raw_hour'] = matrix.index.hour
    
    matrix['hour_sin'] = np.sin(2 * np.pi * matrix['raw_hour'] / 24.0)
    matrix['hour_cos'] = np.cos(2 * np.pi * matrix['raw_hour'] / 24.0)

    matrix.dropna(inplace=True)

    final_features = matrix.drop(columns=['open','high','low','raw_hour','high_low_span'])

    return final_features 

final_feature_matrix = engineer_stationary_features(raw_eurusd_df)
print("Pipeline complete! Displaying top 3 rows of processed ML input matrix:")
print(final_feature_matrix[['close', 'log_return', 'volatility_atr', 'hour_sin', 'hour_cos']].head(3))

Pipeline complete! Displaying top 3 rows of processed ML input matrix:
                       close  log_return  volatility_atr  hour_sin  hour_cos
time                                                                        
2025-12-19 22:00:00  1.17097   -0.001178        0.001249 -0.500000  0.866025
2025-12-19 23:00:00  1.17091   -0.000051        0.001181 -0.258819  0.965926
2025-12-22 00:00:00  1.17106    0.000128        0.001151  0.000000  1.000000


 This function contains the loop that steps through your historical rows, projects the dynamic ATR brackets forward in time, and extracts the target ground-truth label.

In [4]:
def generate_triple_barrier_labels(df: pd.DataFrame, pt_mult: float = 2.0, sl_mult: float = 1.0, max_bars: int = 24) -> pd.Series:

    labels = np.zeros(len(df))
    
    close_prices = df['close'].values
    atr_values = df['volatility_atr'].values
    
    total_rows = len(df)
    
    for t in range(total_rows - max_bars):
        current_close = close_prices[t]
        current_atr = atr_values[t]
        
        upper_barrier = current_close + (current_atr * pt_mult)
        lower_barrier = current_close - (current_atr * sl_mult)
        
        forward_path = close_prices[t + 1 : t + 1 + max_bars]
        
        profit_crossings = np.where(forward_path >= upper_barrier)[0]
        loss_crossings = np.where(forward_path <= lower_barrier)[0]
        
        first_profit_idx = profit_crossings[0] if len(profit_crossings) > 0 else None
        first_loss_idx = loss_crossings[0] if len(loss_crossings) > 0 else None
        
        if first_profit_idx is not None and first_loss_idx is not None:
            if first_profit_idx < first_loss_idx:
                labels[t] = 1  
            else:
                labels[t] = -1  
        elif first_profit_idx is not None:
            labels[t] = 1      
        elif first_loss_idx is not None:
            labels[t] = -1      
        else:
            labels[t] = 0       
            
    return pd.Series(labels, index=df.index, name='target')

target_labels = generate_triple_barrier_labels(final_feature_matrix, pt_mult=2.0, sl_mult=1.0, max_bars=24)

final_feature_matrix['target'] = target_labels
clean_ml_matrix = final_feature_matrix.iloc[:-24].copy()

print(f"Labeling complete! Processed Dataset Shape: {clean_ml_matrix.shape}")


Labeling complete! Processed Dataset Shape: (4963, 7)


In [5]:
class_counts = clean_ml_matrix['target'].value_counts()
print("--- TARGET DISTRIBUTION BALANCE ---")
print(f"Take Profit (+1.0): {class_counts.get(1.0, 0)} trades")
print(f"Stop Loss (-1.0):   {class_counts.get(-1.0, 0)} trades")
print(f"Time Expiry (0.0):  {class_counts.get(0.0, 0)} trades\n")

class_percentages = clean_ml_matrix['target'].value_counts(normalize=True) * 100
print("--- PERCENTAGE PROFILE ---")
print(f"Take Profit: {class_percentages.get(1.0, 0):.2f}%")
print(f"Stop Loss:   {class_percentages.get(-1.0, 0):.2f}%")
print(f"Time Expiry: {class_percentages.get(0.0, 0):.2f}%")

assert class_counts.get(0.0, 0) < (len(clean_ml_matrix) * 0.85), "Warning: Over 85% of your data is marked as Time Expiry. Your ATR barriers may be too wide!"

assert set(clean_ml_matrix['target'].unique()).issubset({-1.0, 0.0, 1.0}), "Error: Invalid classes discovered in target array!"

print("Sprint 2 Data Target Assembly complete! Ready for Purged Cross-Validation.")



--- TARGET DISTRIBUTION BALANCE ---
Take Profit (+1.0): 1446 trades
Stop Loss (-1.0):   3176 trades
Time Expiry (0.0):  341 trades

--- PERCENTAGE PROFILE ---
Take Profit: 29.14%
Stop Loss:   63.99%
Time Expiry: 6.87%
Sprint 2 Data Target Assembly complete! Ready for Purged Cross-Validation.


We will take the first 80% of our timeline for training and reserve the final 20% for testing, adding a 24-hour purge wall between them.

In [6]:
def create_purged_train_test_split(df: pd.DataFrame, target_col: str, max_bars: int = 24, train_ratio: float = 0.8):
    total_rows = len(df)

    split_index = int(total_rows * train_ratio)

    train_df = df.iloc[:split_index].copy()
    test_df = df.iloc[split_index + max_bars:].copy()

    feature_cols = ['log_return','volatility_atr','hour_cos','hour_sin',]

    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    
    X_test = test_df[feature_cols]
    y_test = test_df[target_col]

    print("--- PURGED SPLIT VERIFICATION ---")
    print(f"Training Features Shape (X_train): {X_train.shape}")
    print(f"Testing Features Shape  (X_test):  {X_test.shape}")
    print(f"Data Points Purged in Buffer Gap:   {max_bars} bars\n")

    return X_train,y_train,X_test,y_test 

    
X_train, y_train, X_test,y_test = create_purged_train_test_split(clean_ml_matrix,target_col='target')


--- PURGED SPLIT VERIFICATION ---
Training Features Shape (X_train): (3970, 4)
Testing Features Shape  (X_test):  (969, 4)
Data Points Purged in Buffer Gap:   24 bars



This cell is to handle the Imbalance that arose when we ivestigates in the 2nd cell

In [7]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train)

print("Sample weights generated successfully!")
print(f"First 5 row penalization values: {sample_weights_train[:5]}")


Sample weights generated successfully!
First 5 row penalization values: [1.1437626 1.1437626 1.1437626 1.1437626 1.1437626]


Going to initilize my first basse classifications model. Deliverables: I have finished handling and peforming feature Engineering to the data now lets go to the first base model.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score 

xgb_analyst = XGBClassifier(
    n_estimators=150,
    n_estimators=150,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    random_state=42,
    tree_method='hist' 
)

print("Training XGBoost on our sequential data split....")

# Fitting the Training data to the XgBoost 

xgb_analyst.fit(
    X_train,
    y_train,
    sample_weights=sample_weights_train
)

print("**DONE** XGBOOST Training Complete")